## FINAL WITHOUT NEXT PAGE

In [26]:
# import os
# import re
# import time
# import string
# import paramiko
# import pandas as pd
# from datetime import datetime
# from openpyxl.styles import Font, Alignment

# # ==============================
# # SSH Configuration
# # ==============================
# SAT_HOST = "172.16.70.39"
# SAT_PORT = 5022
# SAT_USERNAME = "dadmin"
# SAT_PASSWORD = "Avaya@123"
# COMMAND = "status trunk 3"

# REPORT_DIR = "reports"
# os.makedirs(REPORT_DIR, exist_ok=True)

# # ==============================
# # Utility Functions
# # ==============================
# def clean_output(output):
#     """Remove ANSI escape codes and control characters."""
#     output = re.sub(r'\x1B[@-_][0-?]*[ -/]*[@-~]', '', output)
#     output = re.sub(r'\r', '', output)
#     output = re.sub(r'(?m)^Command:.*$', '', output)
#     output = ''.join(ch for ch in output if ch in string.printable or ch == '\n')
#     output = re.sub(r'\n{2,}', '\n', output)
#     return output.strip()

# def auth_handler(title, instructions, prompt_list):
#     return [SAT_PASSWORD if 'Password' in p[0] else '' for p in prompt_list]

# # ==============================
# # SSH Command Execution (Same as Monitor)
# # ==============================
# def run_avaya_command():
#     """Connect to Avaya SAT, run command, return cleaned output."""
#     transport = paramiko.Transport((SAT_HOST, SAT_PORT))
#     transport.connect()
#     transport.auth_interactive(SAT_USERNAME, auth_handler)

#     channel = transport.open_session()
#     channel.get_pty()
#     channel.invoke_shell()
#     time.sleep(2)

#     # Handle banner prompt
# # Wait for Terminal Type prompt properly
#     banner_buffer = ""
#     timeout_start = time.time()
#     while True:
#         time.sleep(0.5)
#         if channel.recv_ready():
#             chunk = channel.recv(4096).decode(errors="ignore")
#             banner_buffer += chunk
#             if "Terminal Type" in banner_buffer:
#                 print("🖥️ Terminal Type prompt detected — sending 'VT220'")
#                 channel.send("VT220\n")
#                 time.sleep(2)
#                 break
#         if time.time() - timeout_start > 15:
#             raise RuntimeError("Timeout waiting for Terminal Type prompt")


#     # Enter SAT shell
#     channel.send("sat\n")
#     time.sleep(2)
#     if channel.recv_ready():
#         channel.recv(4096)

#     print(f"→ Executing: {COMMAND}")
#     channel.send(COMMAND + "\n")
#     time.sleep(6)

#     output = ""
#     while channel.recv_ready():
#         output += channel.recv(8192).decode(errors='ignore')
#         time.sleep(0.3)

#     transport.close()
#     return clean_output(output)

# # ==============================
# # Parser (5 Columns)
# # ==============================
# def parse_status_trunk(output):
#     """
#     Parse 'status trunk' output into:
#     Member | Port | Service State | Mtce Busy | Connected Ports
#     """
#     output = output.replace("Service State      Mtce", "Service State   Mtce Busy")
#     output = re.sub(r"[<>]\d+", "", output)  # clean control chars

#     lines = [l.strip() for l in output.splitlines() if re.match(r"^\d{4}/\d{4}", l)]
#     rows = []

#     for line in lines:
#         # split "0003/0001T000011 in-service/idle   no"
#         m = re.match(r"(?P<Member>\d{4}/\d{4})T(?P<Port>\d+)\s+(?P<ServiceState>[a-zA-Z/\-]+)\s+(?P<MtceBusy>\w+)", line)
#         if m:
#             rows.append({
#                 "Member": m.group("Member"),
#                 "Port": f"T{m.group('Port')}",
#                 "Service State": m.group("ServiceState"),
#                 "Mtce Busy": m.group("MtceBusy"),
#                 "Connected Ports": ""  # none shown in your CM output
#             })

#     df = pd.DataFrame(rows, columns=["Member", "Port", "Service State", "Mtce Busy", "Connected Ports"])
#     if df.empty:
#         df.loc[0] = ["No data parsed", "", "", "", ""]
#     else:
#         print(f"✅ Parsed {len(df)} rows successfully.")
#     return df



# # ==============================
# # Main Execution
# # ==============================
# def main():
#     timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#     safe_cmd = COMMAND.replace(" ", "_").replace("/", "-")
#     excel_file = os.path.join(REPORT_DIR, f"{safe_cmd}_{timestamp}.xlsx")

#     try:
#         output = run_avaya_command()
#         print("\n========== CLEANED OUTPUT PREVIEW ==========")
#         print(output[:1500])
#         print("\n========== CLEANED OUTPUT END ==========")

#         df = parse_status_trunk(output)
#         print("DataFrame preview:")
#         print(df.head())
#         print("Rows:", len(df))


#         with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
#             df.to_excel(writer, sheet_name="Status_Trunk", index=False)
#             ws = writer.sheets["Status_Trunk"]

#             bold = Font(bold=True)
#             for cell in ws[1]:
#                 cell.font = bold
#                 cell.alignment = Alignment(horizontal="center", vertical="center")

#             for col in ws.columns:
#                 max_len = max(len(str(cell.value)) for cell in col if cell.value)
#                 ws.column_dimensions[col[0].column_letter].width = max_len + 2

#         print(f"\n✅ Excel report created successfully:\n{excel_file}")

#     except Exception as e:
#         print(f"❌ Error: {e}")

# if __name__ == "__main__":
#     main()


🖥️ Terminal Type prompt detected — sending 'VT220'
→ Executing: status trunk 3

========== CLEANED OUTPUT PREVIEW ==========
status trunk 3                                                                                7status trunk 3 8                                                                                7       Page   18TRUNK GROUP STATUS
Member    Port    Service State      Mtce Connected Ports
                                     Busy
0003/0001T000011 in-service/idle   no                                  
0003/0002T000012 in-service/idle   no                                  
0003/0003T000013 in-service/idle   no                                  
0003/0004T000014 in-service/idle   no                                  
0003/0005T000015 in-service/idle   no                                  
0003/0006T000016 in-service/idle   no                                  
0003/0007T000017 in-service/idle   no                                  
0003/0008T000018 in-service/idle   no       

## FINAL WITH NEXT PAGE

In [27]:
import os
import re
import time
import string
import paramiko
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment

# ==============================
# SSH Configuration
# ==============================
SAT_HOST = "172.16.70.39"
SAT_PORT = 5022
SAT_USERNAME = "dadmin"
SAT_PASSWORD = "Avaya@123"
COMMAND = "status trunk 3"   # change to desired trunk group number

REPORT_DIR = "reports"
os.makedirs(REPORT_DIR, exist_ok=True)

# ==============================
# Utility Functions
# ==============================
def clean_output(output):
    """Remove ANSI escape codes and control characters."""
    output = re.sub(r'\x1B[@-_][0-?]*[ -/]*[@-~]', '', output)
    output = re.sub(r'\r', '', output)
    output = ''.join(ch for ch in output if ch in string.printable or ch == '\n')
    output = re.sub(r'\n{2,}', '\n', output)
    return output.strip()

def auth_handler(title, instructions, prompt_list):
    return [SAT_PASSWORD if 'Password' in p[0] else '' for p in prompt_list]

# ==============================
# Parser
# ==============================
def parse_status_trunk(output):
    """
    Parse 'status trunk' output into:
    Member | Port | Service State | Mtce Busy | Connected Ports
    """
    output = output.replace("Service State      Mtce", "Service State   Mtce Busy")
    output = re.sub(r"[<>]\d+", "", output)  # clean stray cursor codes

    # Extract only member lines (beginning with xxxx/xxxx)
    lines = [l.strip() for l in output.splitlines() if re.match(r"^\d{4}/\d{4}", l)]
    rows = []

    for line in lines:
        m = re.match(
            r"(?P<Member>\d{4}/\d{4})T(?P<Port>\d+)\s+(?P<ServiceState>[a-zA-Z/\-]+)\s+(?P<MtceBusy>\w+)",
            line
        )
        if m:
            rows.append({
                "Member": m.group("Member"),
                "Port": f"T{m.group('Port')}",
                "Service State": m.group("ServiceState"),
                "Mtce Busy": m.group("MtceBusy"),
                "Connected Ports": ""
            })

    df = pd.DataFrame(rows, columns=["Member", "Port", "Service State", "Mtce Busy", "Connected Ports"])
    return df

# ==============================
# Excel Writer (Append)
# ==============================
def append_to_excel(df_page, excel_file):
    """Append a page's DataFrame to an Excel sheet."""
    if not os.path.exists(excel_file):
        with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
            df_page.to_excel(writer, sheet_name="Status_Trunk", index=False)
            ws = writer.sheets["Status_Trunk"]
            bold = Font(bold=True)
            for cell in ws[1]:
                cell.font = bold
                cell.alignment = Alignment(horizontal="center", vertical="center")
    else:
        book = load_workbook(excel_file)
        sheet = book["Status_Trunk"]
        start_row = sheet.max_row + 1
        for _, row in df_page.iterrows():
            for c_idx, val in enumerate(row, start=1):
                sheet.cell(row=start_row, column=c_idx, value=val)
            start_row += 1
        book.save(excel_file)

# ==============================
# Run Command
# ==============================
def run_avaya_command():
    """Connect to Avaya SAT, run the command, handle pagination (F7)."""
    transport = paramiko.Transport((SAT_HOST, SAT_PORT))
    transport.connect()
    transport.auth_interactive(SAT_USERNAME, auth_handler)

    channel = transport.open_session()
    channel.get_pty()
    channel.invoke_shell()
    time.sleep(2)

    # Wait for Terminal Type prompt
    banner_buffer = ""
    timeout_start = time.time()
    while True:
        time.sleep(0.5)
        if channel.recv_ready():
            chunk = channel.recv(4096).decode(errors="ignore")
            banner_buffer += chunk
            if "Terminal Type" in banner_buffer:
                print("🖥️ Terminal Type prompt detected — sending 'VT220'")
                channel.send("VT220\n")
                time.sleep(2)
                break
        if time.time() - timeout_start > 15:
            raise RuntimeError("Timeout waiting for Terminal Type prompt")

    # Enter SAT shell
    channel.send("sat\n")
    time.sleep(2)
    if channel.recv_ready():
        _ = channel.recv(4096)

    # Execute command
    print(f"→ Executing: {COMMAND}")
    channel.send(COMMAND + "\n")
    time.sleep(3)

    # Prepare Excel file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_cmd = COMMAND.replace(" ", "_").replace("/", "-")
    excel_file = os.path.join(REPORT_DIR, f"{safe_cmd}_{timestamp}.xlsx")

    full_output = ""
    page_counter = 1

    while True:
        time.sleep(0.8)
        if channel.recv_ready():
            chunk = channel.recv(65535).decode(errors="ignore")
            full_output += chunk

            if re.search(r"press\s+next\s+page", chunk, re.IGNORECASE):
                print(f"📄 Page {page_counter} detected")
                page_text = clean_output(chunk)
                df_page = parse_status_trunk(page_text)

                if df_page.empty:
                    print("🛑 Empty page detected — stopping pagination.")
                    break

                print(f"✅ Parsed {len(df_page)} rows from page {page_counter}")
                append_to_excel(df_page, excel_file)

                page_counter += 1
                print("➡️ Sending F7 (ESC [18~) to fetch next page...")
                channel.send("\x1b[18~")  # emulate F7
                continue

            if re.search(r"command\s+successfully\s+completed", chunk, re.IGNORECASE):
                print(f"✅ Final page ({page_counter}) detected — appending and exiting.")
                df_page = parse_status_trunk(clean_output(chunk))
                if not df_page.empty:
                    append_to_excel(df_page, excel_file)
                break
        else:
            time.sleep(0.5)

    transport.close()
    print(f"\n✅ Excel report created successfully:\n{os.path.abspath(excel_file)}")

# ==============================
# Main
# ==============================
def main():
    try:
        run_avaya_command()
    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    main()


🖥️ Terminal Type prompt detected — sending 'VT220'
→ Executing: status trunk 3
✅ Final page (1) detected — appending and exiting.

✅ Excel report created successfully:
/Users/sooryaraysam/Checklist Automation/Trunks/reports/status_trunk_3_20251024_112804.xlsx
